# CSCE 636 Project 1: MoE DNN for m-Height (Colab GPU)

**Instructions:**
1. Go to `Runtime → Change runtime type → GPU`
2. Upload your data files: `CSCE-636-Project-1-Train-n_k_m_P` and `CSCE-636-Project-1-Train-mHeights`
3. Run all cells (`Runtime → Run all`)
4. Download `moe_model.pth` and `predicted_mHeights` when done

In [1]:
# ============================================================
# Cell 1: Setup — Install deps, verify GPU, upload data
# ============================================================
!pip install scipy scikit-learn -q

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")
    print("⚠️  NO GPU! Go to: Runtime → Change runtime type → GPU")

print(f"\nUsing device: {device}")

PyTorch version: 2.10.0+cu128
CUDA available: True
✅ GPU: Tesla T4
   Memory: 15.6 GB

Using device: cuda


In [2]:
# ============================================================
# Cell 2: Upload data files (run this, then click upload)
# ============================================================
import os

# Check if files already exist (e.g., mounted from Drive)
x_file = 'CSCE-636-Project-1-Train-n_k_m_P'
y_file = 'CSCE-636-Project-1-Train-mHeights'

if not os.path.exists(x_file) or not os.path.exists(y_file):
    from google.colab import files
    print("Please upload BOTH data files:")
    print(f"  1. {x_file}")
    print(f"  2. {y_file}")
    uploaded = files.upload()
    print(f"\nUploaded {len(uploaded)} files.")
else:
    print("Data files found!")

print(f"  {x_file}: {os.path.exists(x_file)}")
print(f"  {y_file}: {os.path.exists(y_file)}")

Please upload BOTH data files:
  1. CSCE-636-Project-1-Train-n_k_m_P
  2. CSCE-636-Project-1-Train-mHeights


Saving CSCE-636-Project-1-Train-n_k_m_P to CSCE-636-Project-1-Train-n_k_m_P
Saving CSCE-636-Project-1-Train-mHeights to CSCE-636-Project-1-Train-mHeights

Uploaded 2 files.
  CSCE-636-Project-1-Train-n_k_m_P: True
  CSCE-636-Project-1-Train-mHeights: True


In [3]:
# ============================================================
# Cell 3: Load Data
# ============================================================
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from collections import Counter, defaultdict
import time
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

x_file_path = 'CSCE-636-Project-1-Train-n_k_m_P'
y_file_path = 'CSCE-636-Project-1-Train-mHeights'

with open(x_file_path, 'rb') as f:
    X_raw = pickle.load(f)
with open(y_file_path, 'rb') as f:
    y_raw = pickle.load(f)

print(f"Total samples loaded: {len(X_raw)}")
print(f"m-height range: [{min(y_raw):.4f}, {max(y_raw):.4f}]")

param_counts = Counter([(s[0], s[1], s[2]) for s in X_raw])
print("\nSample counts per (n, k, m):")
for params, count in sorted(param_counts.items()):
    print(f"  n={params[0]}, k={params[1]}, m={params[2]}: {count} samples")

Total samples loaded: 96524
m-height range: [2.0000, 6450061.6667]

Sample counts per (n, k, m):
  n=9, k=4, m=2: 11157 samples
  n=9, k=4, m=3: 11003 samples
  n=9, k=4, m=4: 10253 samples
  n=9, k=4, m=5: 6765 samples
  n=9, k=5, m=2: 10803 samples
  n=9, k=5, m=3: 10159 samples
  n=9, k=5, m=4: 6573 samples
  n=9, k=6, m=2: 18898 samples
  n=9, k=6, m=3: 10913 samples


In [4]:
# ============================================================
# Cell 4: LP-Based Data Augmentation
# ============================================================
from scipy.optimize import linprog
from itertools import combinations

def compute_m_height_lp(n, k, m, P):
    """Compute exact m-height via LP algorithm from project description."""
    G = np.hstack([np.eye(k), P])
    max_z = 1.0
    for S in combinations(range(n), m):
        S_set = set(S)
        S_bar = [t for t in range(n) if t not in S_set]
        for j in S:
            c = -G[:, j]
            A_ub = []
            b_ub = []
            for t in S_bar:
                A_ub.append(G[:, t])
                b_ub.append(1.0)
                A_ub.append(-G[:, t])
                b_ub.append(1.0)
            A_ub = np.array(A_ub)
            b_ub = np.array(b_ub)
            result = linprog(c, A_ub=A_ub, b_ub=b_ub, method='highs')
            if result.success:
                z_sj = -result.fun
                if z_sj > max_z:
                    max_z = z_sj
    return max_z

def generate_augmented_samples(n, k, m, num_samples, p_range=10.0):
    """Generate new (P, m-height) samples using the LP algorithm."""
    new_X, new_y = [], []
    for _ in range(num_samples):
        P = np.random.uniform(-p_range, p_range, size=(k, n - k))
        mh = compute_m_height_lp(n, k, m, P)
        if np.isfinite(mh) and mh >= 1.0:
            new_X.append([n, k, m, P])
            new_y.append(mh)
    return new_X, new_y

# Augment under-represented groups to ~15000
target_per_group = 15000
aug_X_all, aug_y_all = [], []

print("Generating augmented samples via LP algorithm...")
print("(Uses CPU — typically 5-15 min on Colab)\n")

for (n_val, k_val, m_val), count in sorted(param_counts.items()):
    deficit = max(0, target_per_group - count)
    if deficit > 0:
        num_to_gen = min(deficit, 3000)
        print(f"  (n={n_val}, k={k_val}, m={m_val}): existing={count}, generating {num_to_gen}...", end=" ", flush=True)
        t0 = time.time()
        new_x, new_y = generate_augmented_samples(n_val, k_val, m_val, num_to_gen, p_range=10.0)
        aug_X_all.extend(new_x)
        aug_y_all.extend(new_y)
        print(f"done ({len(new_y)} valid, {time.time()-t0:.1f}s)")
    else:
        print(f"  (n={n_val}, k={k_val}, m={m_val}): existing={count}, skip")

X_combined = X_raw + aug_X_all
y_combined = y_raw + aug_y_all

print(f"\nOriginal: {len(X_raw)}, Augmented: {len(aug_X_all)}, Total: {len(X_combined)}")

combined_counts = Counter([(s[0], s[1], s[2]) for s in X_combined])
print("\nUpdated counts:")
for params, count in sorted(combined_counts.items()):
    print(f"  n={params[0]}, k={params[1]}, m={params[2]}: {count}")

Generating augmented samples via LP algorithm...
(Uses CPU — typically 5-15 min on Colab)

  (n=9, k=4, m=2): existing=11157, generating 3000... done (3000 valid, 379.7s)
  (n=9, k=4, m=3): existing=11003, generating 3000... done (3000 valid, 1321.4s)
  (n=9, k=4, m=4): existing=10253, generating 3000... done (3000 valid, 2627.4s)
  (n=9, k=4, m=5): existing=6765, generating 3000... done (3000 valid, 3355.9s)
  (n=9, k=5, m=2): existing=10803, generating 3000... done (3000 valid, 413.3s)
  (n=9, k=5, m=3): existing=10159, generating 3000... done (3000 valid, 1456.4s)
  (n=9, k=5, m=4): existing=6573, generating 3000... done (3000 valid, 2814.3s)
  (n=9, k=6, m=2): existing=18898, skip
  (n=9, k=6, m=3): existing=10913, generating 3000... done (3000 valid, 1409.6s)

Original: 96524, Augmented: 24000, Total: 120524

Updated counts:
  n=9, k=4, m=2: 14157
  n=9, k=4, m=3: 14003
  n=9, k=4, m=4: 13253
  n=9, k=4, m=5: 9765
  n=9, k=5, m=2: 13803
  n=9, k=5, m=3: 13159
  n=9, k=5, m=4: 9573

In [ ]:
# ============================================================
# Cell 5: Feature Engineering (enhanced 74-dim features)
# ============================================================

def extract_features(sample):
    """Extract 74-dim feature vector from [n, k, m, P]."""
    n, k, m, P = sample
    max_k, max_nk, max_sv = 6, 5, 5

    # --- Padded P matrix ---
    P_padded = np.zeros((max_k, max_nk))
    P_padded[:k, :n-k] = P
    P_flat = P_padded.flatten()  # 30

    # --- Row norms of P ---
    row_norms = np.zeros(max_k)
    for i in range(k):
        row_norms[i] = np.linalg.norm(P[i, :])  # 6

    # --- Column norms of P ---
    col_norms = np.zeros(max_nk)
    for j in range(n - k):
        col_norms[j] = np.linalg.norm(P[:, j])  # 5

    # --- Singular values of P ---
    sv = np.zeros(max_sv)
    try:
        singular_values = np.linalg.svd(P, compute_uv=False)
        sv[:len(singular_values)] = np.sort(singular_values)[::-1]
    except:
        pass  # 5

    # --- Basic stats of P ---
    frob_norm = np.linalg.norm(P, 'fro')
    max_abs = np.max(np.abs(P))
    mean_abs = np.mean(np.abs(P))
    std_abs = np.std(np.abs(P))
    min_abs = np.min(np.abs(P))

    try:
        cond = np.log1p(np.linalg.cond(P))
    except:
        cond = 0.0
    if not np.isfinite(cond):
        cond = 50.0

    mean_row = np.mean(row_norms[:k]) if k > 0 else 1.0
    mean_col = np.mean(col_norms[:n-k]) if (n-k) > 0 else 1.0
    ratio = mean_row / (mean_col + 1e-8)

    # --- NEW: G=[I|P] column norms (key for m-height!) ---
    G = np.hstack([np.eye(k), P])
    g_col_norms = np.zeros(9)  # n=9 always
    for j in range(n):
        g_col_norms[j] = np.linalg.norm(G[:, j])
    g_col_max = np.max(g_col_norms)
    g_col_min = np.min(g_col_norms[:n])
    g_col_mean = np.mean(g_col_norms[:n])
    g_col_std = np.std(g_col_norms[:n])

    # --- NEW: Rank and effective rank ---
    try:
        rank = np.linalg.matrix_rank(P)
    except:
        rank = min(k, n-k)
    effective_rank = np.sum(sv > 1e-6)

    # --- NEW: Singular value ratios ---
    sv_ratio = sv[0] / (sv[min(k, n-k)-1] + 1e-10) if sv[min(k, n-k)-1] > 1e-10 else 100.0
    sv_sum = np.sum(sv)
    sv_energy_ratio = sv[0] / (sv_sum + 1e-10)  # energy in first SV

    # --- NEW: Cross-column interactions of P ---
    try:
        PtP = P.T @ P
        ptP_trace = np.trace(PtP)
        ptP_frob = np.linalg.norm(PtP, 'fro')
    except:
        ptP_trace = frob_norm ** 2
        ptP_frob = frob_norm ** 2

    # --- NEW: Determinant-based features ---
    try:
        if k <= n-k:
            det_val = np.log1p(abs(np.linalg.det(P[:k, :k])))
        else:
            det_val = np.log1p(abs(np.linalg.det(P[:min(k,n-k), :min(k,n-k)])))
    except:
        det_val = 0.0
    if not np.isfinite(det_val):
        det_val = 0.0

    return np.concatenate([
        [n, k, m],                     # 3
        P_flat,                         # 30
        row_norms,                      # 6
        col_norms,                      # 5
        sv,                             # 5
        [frob_norm, max_abs, mean_abs, std_abs, min_abs],  # 5
        [cond, ratio],                  # 2
        g_col_norms,                    # 9
        [g_col_max, g_col_min, g_col_mean, g_col_std],  # 4
        [rank, effective_rank],         # 2
        [sv_ratio, sv_sum, sv_energy_ratio],  # 3
        [ptP_trace, ptP_frob],          # 2
        [det_val],                      # 1
    ])  # total: 3+30+6+5+5+5+2+9+4+2+3+2+1 = 77
    # (some may vary, we'll check FEATURE_DIM)


def preprocess_all(X_list, y_list):
    """Preprocess all samples into feature array and log2 target."""
    features = [extract_features(s) for s in X_list]
    X_array = np.array(features, dtype=np.float32)
    y_array = np.log2(np.array(y_list, dtype=np.float32)).reshape(-1, 1)
    return X_array, y_array


X_array, y_log2 = preprocess_all(X_combined, y_combined)
FEATURE_DIM = X_array.shape[1]
print(f"Feature dim: {FEATURE_DIM}")
print(f"Total samples: {X_array.shape[0]}")
print(f"log2(y) range: [{y_log2.min():.4f}, {y_log2.max():.4f}]")

# Sanity check for NaN/Inf
nan_count = np.isnan(X_array).sum()
inf_count = np.isinf(X_array).sum()
print(f"NaN features: {nan_count}, Inf features: {inf_count}")
if nan_count > 0 or inf_count > 0:
    X_array = np.nan_to_num(X_array, nan=0.0, posinf=100.0, neginf=-100.0)
    print("  → Cleaned NaN/Inf values.")

Feature dim: 54
Total samples: 120524
log2(y) range: [0.0000, 22.6209]


In [ ]:
# ============================================================
# Cell 6: MoE Dataset Preparation (GPU-optimized loaders)
# ============================================================

class MHeightDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Re-confirm device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

group_keys = sorted(set((s[1], s[2]) for s in X_combined))
print(f"MoE Groups: {group_keys}")

group_data = {}
batch_size = 256
use_pin = (device.type == "cuda")  # pin_memory for faster GPU transfer

for (k_val, m_val) in group_keys:
    indices = [i for i, s in enumerate(X_combined) if s[1] == k_val and s[2] == m_val]
    X_group = X_array[indices]
    y_group = y_log2[indices]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_group)

    np.random.seed(42)
    perm = np.random.permutation(len(X_scaled))
    split = int(0.9 * len(perm))
    train_idx, val_idx = perm[:split], perm[split:]

    train_ds = MHeightDataset(X_scaled[train_idx], y_group[train_idx])
    val_ds = MHeightDataset(X_scaled[val_idx], y_group[val_idx])

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        pin_memory=use_pin, num_workers=2
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        pin_memory=use_pin, num_workers=2
    )

    group_data[(k_val, m_val)] = {
        'scaler': scaler,
        'train_loader': train_loader,
        'val_loader': val_loader,
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'feature_dim': X_scaled.shape[1],
    }

    print(f"  (k={k_val}, m={m_val}): {len(indices)} total, "
          f"{len(train_idx)} train, {len(val_idx)} val")

In [7]:
# ============================================================
# Cell 7: Residual DNN Architecture
# ============================================================

class ResidualBlock(nn.Module):
    """Residual block with skip connection."""
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.block(x) + x)


class ExpertDNN(nn.Module):
    """Residual DNN expert for a single (k, m) group."""
    def __init__(self, input_dim, hidden_dim=256, num_blocks=3, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)]
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.res_blocks(x)
        return self.output_head(x)


test_model = ExpertDNN(input_dim=FEATURE_DIM).to(device)
total_params = sum(p.numel() for p in test_model.parameters())
print(f"Params per expert: {total_params:,}")
print(f"Total params (9 experts): {total_params * 9:,}")
print(f"Model is on: {next(test_model.parameters()).device}")
del test_model

Params per expert: 428,929
Total params (9 experts): 3,860,361
Model is on: cuda:0


In [ ]:
# ============================================================
# Cell 8: Train All 9 Expert DNNs on GPU (Improved)
# ============================================================
# Key improvements:
# - Adaptive architecture: hard groups get bigger models
# - Huber loss: robust to outliers in log-space
# - Gradient clipping for stability
# - Best-of-N runs: train each expert N times, keep best
# - SWA (Stochastic Weight Averaging) for better generalization
# ============================================================

# Per-group hyperparameters: hard groups get more capacity
GROUP_HPARAMS = {
    # (k, m): (hidden_dim, num_blocks, dropout, lr, epochs, patience, num_runs)
    (4, 2): (256, 3, 0.15, 1e-3, 300, 25, 2),
    (4, 3): (256, 3, 0.15, 1e-3, 300, 25, 2),
    (4, 4): (384, 4, 0.12, 8e-4, 400, 30, 3),
    (4, 5): (512, 5, 0.10, 5e-4, 500, 35, 3),  # hard
    (5, 2): (256, 3, 0.15, 1e-3, 300, 25, 2),
    (5, 3): (384, 4, 0.12, 8e-4, 400, 30, 2),
    (5, 4): (512, 5, 0.10, 5e-4, 500, 35, 3),  # hard
    (6, 2): (256, 3, 0.15, 1e-3, 300, 25, 2),
    (6, 3): (512, 5, 0.10, 5e-4, 500, 35, 3),  # hard
}

def train_expert_v2(group_key, group_info, device, hparams):
    """Train one expert DNN with adaptive hyperparameters, best-of-N runs."""
    k_val, m_val = group_key
    hidden_dim, num_blocks, dropout, lr, num_epochs, patience, num_runs = hparams

    print(f"\n{'='*60}")
    print(f"Training Expert (k={k_val}, m={m_val}) on {device}")
    print(f"  Arch: hidden={hidden_dim}, blocks={num_blocks}, dropout={dropout}")
    print(f"  Train: {group_info['train_size']}, Val: {group_info['val_size']}")
    print(f"  Epochs: {num_epochs}, Patience: {patience}, Runs: {num_runs}")
    print(f"{'='*60}")

    global_best_val = float('inf')
    global_best_state = None
    global_best_t_hist = None
    global_best_v_hist = None

    for run in range(num_runs):
        if num_runs > 1:
            print(f"\n  --- Run {run+1}/{num_runs} ---")

        torch.manual_seed(42 + run * 777)
        np.random.seed(42 + run * 777)

        model = ExpertDNN(
            input_dim=group_info['feature_dim'],
            hidden_dim=hidden_dim,
            num_blocks=num_blocks,
            dropout=dropout
        ).to(device)

        criterion = nn.SmoothL1Loss(beta=0.5)  # Huber loss, robust to outliers
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=40, T_mult=2, eta_min=1e-7
        )

        best_val_loss = float('inf')
        patience_counter = 0
        train_losses, val_losses = [], []
        best_state = None

        for epoch in range(num_epochs):
            # Training
            model.train()
            train_loss = 0.0
            for bx, by in group_info['train_loader']:
                bx = bx.to(device, non_blocking=True)
                by = by.to(device, non_blocking=True)
                pred = model(bx)
                loss = criterion(pred, by)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss += loss.item() * bx.size(0)
            train_loss /= group_info['train_size']

            # Validation (use MSE for consistent comparison with grading cost)
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for bx, by in group_info['val_loader']:
                    bx = bx.to(device, non_blocking=True)
                    by = by.to(device, non_blocking=True)
                    pred = model(bx)
                    loss = nn.functional.mse_loss(pred, by)
                    val_loss += loss.item() * bx.size(0)
            val_loss /= group_info['val_size']

            scheduler.step()
            train_losses.append(train_loss)
            val_losses.append(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1

            if (epoch + 1) % 100 == 0 or epoch == 0:
                cur_lr = optimizer.param_groups[0]['lr']
                print(f"    Epoch [{epoch+1:3d}/{num_epochs}] | "
                      f"Train: {train_loss:.6f} | Val(MSE): {val_loss:.6f} | "
                      f"Best: {best_val_loss:.6f} | LR: {cur_lr:.2e}")

            if patience_counter >= patience:
                print(f"    Early stop at epoch {epoch+1}")
                break

        print(f"    Run {run+1} Best Val: {best_val_loss:.6f}")

        if best_val_loss < global_best_val:
            global_best_val = best_val_loss
            global_best_state = best_state
            global_best_t_hist = train_losses
            global_best_v_hist = val_losses

    model = ExpertDNN(
        input_dim=group_info['feature_dim'],
        hidden_dim=hidden_dim,
        num_blocks=num_blocks,
        dropout=dropout
    ).to(device)
    model.load_state_dict(global_best_state)
    model.to(device)
    print(f"  ✓ Global Best Val Loss: {global_best_val:.6f}")
    return model, global_best_val, global_best_t_hist, global_best_v_hist


# --- Train all 9 experts ---
print(f"\n{'#'*60}")
print(f"# TRAINING 9 EXPERTS ON: {device}")
print(f"# With adaptive per-group hyperparameters")
print(f"{'#'*60}")

start_time = time.time()
experts = {}
all_train_histories = {}
expert_hparams = {}  # store hparams for saving

for group_key in group_keys:
    hparams = GROUP_HPARAMS.get(group_key, (256, 3, 0.15, 1e-3, 300, 25, 2))
    expert_hparams[group_key] = hparams
    t0 = time.time()
    model, best_loss, t_hist, v_hist = train_expert_v2(
        group_key, group_data[group_key], device, hparams
    )
    experts[group_key] = model
    all_train_histories[group_key] = (t_hist, v_hist)
    print(f"  ⏱ {time.time()-t0:.1f}s")

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"ALL 9 EXPERTS TRAINED in {total_time:.1f}s")
print(f"{'='*60}")
for gk in group_keys:
    print(f"  (k={gk[0]},m={gk[1]}): best val = {min(all_train_histories[gk][1]):.6f}")

if device.type == 'cuda':
    print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated()/1e6:.1f} MB")


############################################################
# TRAINING 9 EXPERTS ON: cuda
############################################################

Training Expert (k=4, m=2) on cuda
  Train: 12741, Val: 1416
  Epoch [  1/300] | Train: 3.580750 | Val: 0.685899 | Best: 0.685899 | LR: 0.000997
  Early stop at epoch 48
  ✓ Best Val Loss: 0.240465
  ⏱ 26.0s

Training Expert (k=4, m=3) on cuda
  Train: 12602, Val: 1401
  Epoch [  1/300] | Train: 3.808964 | Val: 1.060969 | Best: 1.060969 | LR: 0.000997
  Epoch [ 50/300] | Train: 0.292138 | Val: 0.303841 | Best: 0.289891 | LR: 0.000750
  Epoch [100/300] | Train: 0.275082 | Val: 0.285053 | Best: 0.270878 | LR: 0.000983
  Early stop at epoch 107
  ✓ Best Val Loss: 0.270878
  ⏱ 44.6s

Training Expert (k=4, m=4) on cuda
  Train: 11927, Val: 1326
  Epoch [  1/300] | Train: 9.093049 | Val: 1.548256 | Best: 1.548256 | LR: 0.000997
  Epoch [ 50/300] | Train: 0.658919 | Val: 0.694436 | Best: 0.627500 | LR: 0.000750
  Early stop at epoch 96
  ✓ B

In [ ]:
# ============================================================
# Cell 9: Plot Training Curves
# ============================================================

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Training & Validation Loss per Expert (k, m)', fontsize=14)

for idx, gk in enumerate(group_keys):
    ax = axes[idx // 3][idx % 3]
    t_hist, v_hist = all_train_histories[gk]
    ax.plot(t_hist, label='Train', linewidth=1)
    ax.plot(v_hist, label='Val', linewidth=1)
    ax.set_title(f'k={gk[0]}, m={gk[1]} (best={min(v_hist):.4f})')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE (log2)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 10: Evaluate on Validation Set
# ============================================================

print("Per-Expert Validation Performance:")
print(f"{'Group':<12} {'Avg Cost':>10} {'RMSE_log2':>10} {'Samples':>8}")
print("-" * 45)

total_cost_sum = 0.0
total_val_count = 0

for gk in group_keys:
    model = experts[gk]
    model.eval()
    preds_list, true_list = [], []

    with torch.no_grad():
        for bx, by in group_data[gk]['val_loader']:
            bx = bx.to(device, non_blocking=True)
            p = model(bx).cpu().numpy()
            preds_list.append(p)
            true_list.append(by.numpy())

    preds = np.concatenate(preds_list).flatten()
    trues = np.concatenate(true_list).flatten()

    costs = (trues - preds) ** 2
    avg_cost = costs.mean()
    rmse = np.sqrt(avg_cost)
    n_samples = len(trues)

    total_cost_sum += costs.sum()
    total_val_count += n_samples

    print(f"(k={gk[0]},m={gk[1]})  {avg_cost:10.6f} {rmse:10.6f} {n_samples:8d}")

overall_cost = total_cost_sum / total_val_count
print("-" * 45)
print(f"{'OVERALL':<12} {overall_cost:10.6f} {np.sqrt(overall_cost):10.6f} {total_val_count:8d}")
print(f"\n★ Overall average grading cost: {overall_cost:.6f}")

In [ ]:
# ============================================================
# Cell 10b: Fast GPU Ensemble for Weak Groups (NO LP needed)
# ============================================================
# Instead of slow LP augmentation, train additional ensemble
# models on the EXISTING data with different seeds, architectures,
# and learning rates. Averaging reduces variance → lower cost.
# ============================================================

# Identify weak groups from Cell 10 evaluation
weak_groups = []
COST_THRESHOLD = 1.0  # groups with avg cost > this are "weak"

# Re-evaluate to find weak groups automatically
print("Identifying weak groups (cost > {:.1f})...".format(COST_THRESHOLD))
group_costs = {}
for gk in group_keys:
    model = experts[gk]
    model.eval()
    preds_list, true_list = [], []
    with torch.no_grad():
        for bx, by in group_data[gk]['val_loader']:
            bx = bx.to(device, non_blocking=True)
            p = model(bx).cpu().numpy()
            preds_list.append(p)
            true_list.append(by.numpy())
    preds = np.concatenate(preds_list).flatten()
    trues = np.concatenate(true_list).flatten()
    avg_cost = ((trues - preds) ** 2).mean()
    group_costs[gk] = avg_cost
    if avg_cost > COST_THRESHOLD:
        weak_groups.append(gk)
    print(f"  (k={gk[0]},m={gk[1]}): cost={avg_cost:.4f} {'★ WEAK' if avg_cost > COST_THRESHOLD else '✓ ok'}")

if not weak_groups:
    print("\n✅ No weak groups! All costs are below threshold.")
    print("Skipping ensemble training.")
    ENSEMBLE_SIZE = 0
    weak_ensembles = {}
    weak_scalers = {}
    use_ensemble_for = []
else:
    print(f"\nWeak groups to improve: {weak_groups}")

    # Train ensemble of 5 models per weak group (fast on GPU, no LP needed)
    ENSEMBLE_SIZE = 5
    weak_ensembles = {}
    weak_scalers = {}

    # Different architectures for diversity in the ensemble
    ENSEMBLE_CONFIGS = [
        # (hidden, blocks, dropout, lr, seed_offset)
        (512, 5, 0.08, 3e-4, 100),
        (512, 6, 0.10, 5e-4, 200),
        (384, 5, 0.08, 4e-4, 300),
        (512, 4, 0.12, 3e-4, 400),
        (640, 5, 0.10, 2e-4, 500),
    ]

    for gk in weak_groups:
        k_val, m_val = gk
        info = group_data[gk]
        weak_scalers[gk] = info['scaler']
        ensemble_models = []

        print(f"\n{'='*60}")
        print(f"ENSEMBLE for (k={k_val}, m={m_val}) — {ENSEMBLE_SIZE} diverse models on {device}")
        print(f"  Data: {info['train_size']} train, {info['val_size']} val")
        print(f"  Original cost: {group_costs[gk]:.4f}")
        print(f"{'='*60}")

        for ens_idx, (h_dim, n_blk, drop, lr, seed_off) in enumerate(ENSEMBLE_CONFIGS[:ENSEMBLE_SIZE]):
            print(f"\n  --- Model {ens_idx+1}/{ENSEMBLE_SIZE} (h={h_dim}, b={n_blk}, d={drop}, lr={lr}) ---")

            torch.manual_seed(seed_off + gk[0] * 100 + gk[1] * 10)
            np.random.seed(seed_off + gk[0] * 100 + gk[1] * 10)

            model = ExpertDNN(
                input_dim=info['feature_dim'],
                hidden_dim=h_dim,
                num_blocks=n_blk,
                dropout=drop
            ).to(device)

            criterion = nn.SmoothL1Loss(beta=0.5)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=50, T_mult=2, eta_min=1e-7
            )

            best_val_loss = float('inf')
            patience_counter = 0
            max_epochs = 600
            patience_limit = 40

            for epoch in range(max_epochs):
                model.train()
                for bx, by in info['train_loader']:
                    bx = bx.to(device, non_blocking=True)
                    by = by.to(device, non_blocking=True)
                    pred = model(bx)
                    loss = criterion(pred, by)
                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for bx, by in info['val_loader']:
                        bx = bx.to(device, non_blocking=True)
                        by = by.to(device, non_blocking=True)
                        pred = model(bx)
                        loss = nn.functional.mse_loss(pred, by)
                        val_loss += loss.item() * bx.size(0)
                val_loss /= info['val_size']

                scheduler.step()

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                else:
                    patience_counter += 1

                if (epoch + 1) % 100 == 0 or epoch == 0:
                    print(f"    Epoch [{epoch+1:3d}/{max_epochs}] Val: {val_loss:.6f} Best: {best_val_loss:.6f}")

                if patience_counter >= patience_limit:
                    print(f"    Early stop at epoch {epoch+1}")
                    break

            model.load_state_dict(best_state)
            model.to(device)
            model.eval()
            ensemble_models.append(model)
            print(f"    ✓ Best Val: {best_val_loss:.6f}")

        weak_ensembles[gk] = ensemble_models

    print(f"\n{'='*60}")
    print("ENSEMBLE TRAINING COMPLETE")
    print(f"{'='*60}")
    for gk, models in weak_ensembles.items():
        print(f"  (k={gk[0]}, m={gk[1]}): {len(models)} ensemble models ready")

    if device.type == 'cuda':
        print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated()/1e6:.1f} MB")

In [ ]:
# ============================================================
# Cell 10c: Evaluate Ensembles vs Original Experts
# ============================================================

# Safe check: if no weak groups were found, skip evaluation
_weak_groups = weak_groups if 'weak_groups' in dir() else []
_weak_ensembles = weak_ensembles if 'weak_ensembles' in dir() else {}
_weak_scalers = weak_scalers if 'weak_scalers' in dir() else {}

if not _weak_groups or not _weak_ensembles:
    print("No ensemble models to evaluate. Using original experts only.")
    use_ensemble_for = []
else:
    print("Comparison: Original Expert vs Ensemble (averaged)")
    print(f"{'Group':<12} {'Old Cost':>10} {'Ens Cost':>10} {'Improvement':>12}")
    print("-" * 50)

    improved_costs = {}

    for gk in _weak_groups:
        k_val, m_val = gk
        info = group_data[gk]

        # Ensemble prediction (average of N models using SAME scaler as original)
        ensemble_preds_list = []
        for ens_model in _weak_ensembles[gk]:
            ens_model.eval()
            preds_one = []
            with torch.no_grad():
                for bx, by in info['val_loader']:
                    bx = bx.to(device, non_blocking=True)
                    p = ens_model(bx).cpu().numpy()
                    preds_one.append(p)
            ensemble_preds_list.append(np.concatenate(preds_one).flatten())

        true_vals = []
        for bx, by in info['val_loader']:
            true_vals.append(by.numpy())
        trues = np.concatenate(true_vals).flatten()

        # Also include the original expert's prediction in the average
        orig_model = experts[gk]
        orig_model.eval()
        orig_preds = []
        with torch.no_grad():
            for bx, by in info['val_loader']:
                bx = bx.to(device, non_blocking=True)
                orig_preds.append(orig_model(bx).cpu().numpy())
        orig_pred_arr = np.concatenate(orig_preds).flatten()

        # Average: original + all ensemble models
        all_pred_arrays = [orig_pred_arr] + ensemble_preds_list
        ensemble_avg = np.mean(all_pred_arrays, axis=0)

        new_costs = (trues - ensemble_avg) ** 2
        new_avg_cost = new_costs.mean()

        old_costs = (trues - orig_pred_arr) ** 2
        old_avg_cost = old_costs.mean()

        improvement = old_avg_cost - new_avg_cost
        improved_costs[gk] = {
            'new_cost': new_avg_cost,
            'old_cost': old_avg_cost,
            'new_cost_sum': new_costs.sum(),
            'n_samples': len(trues),
        }

        status = "✓ IMPROVED" if improvement > 0 else "✗ worse"
        print(f"(k={k_val},m={m_val})  {old_avg_cost:10.6f} {new_avg_cost:10.6f} {improvement:+10.6f}  {status}")

    # Compute new overall cost
    print(f"\n{'='*50}")
    print("NEW Overall Evaluation:")
    print(f"{'Group':<12} {'Cost':>10} {'Source':>12} {'Samples':>8}")
    print("-" * 46)

    total_cost_sum_new = 0.0
    total_count_new = 0

    for gk in group_keys:
        if gk in improved_costs and improved_costs[gk]['new_cost'] < improved_costs[gk]['old_cost']:
            cost = improved_costs[gk]['new_cost']
            n_samp = improved_costs[gk]['n_samples']
            total_cost_sum_new += improved_costs[gk]['new_cost_sum']
            total_count_new += n_samp
            source = "ENSEMBLE"
        else:
            model = experts[gk]
            model.eval()
            preds_l, trues_l = [], []
            with torch.no_grad():
                for bx, by in group_data[gk]['val_loader']:
                    bx = bx.to(device, non_blocking=True)
                    preds_l.append(model(bx).cpu().numpy())
                    trues_l.append(by.numpy())
            preds_arr = np.concatenate(preds_l).flatten()
            trues_arr = np.concatenate(trues_l).flatten()
            c = ((trues_arr - preds_arr) ** 2)
            cost = c.mean()
            total_cost_sum_new += c.sum()
            total_count_new += len(trues_arr)
            source = "original"
            n_samp = len(trues_arr)

        print(f"(k={gk[0]},m={gk[1]})  {cost:10.6f} {source:>12} {n_samp:8d}")

    new_overall = total_cost_sum_new / total_count_new
    print("-" * 46)
    print(f"{'NEW OVERALL':<12} {new_overall:10.6f}")
    print(f"\n★ Previous overall cost: {overall_cost:.6f}")
    print(f"★ New overall cost:      {new_overall:.6f}")
    print(f"★ Improvement:           {overall_cost - new_overall:+.6f}")

    # Decide which groups to use ensemble for
    use_ensemble_for = []
    for gk in _weak_groups:
        if gk in improved_costs and improved_costs[gk]['new_cost'] < improved_costs[gk]['old_cost']:
            use_ensemble_for.append(gk)
            print(f"\n→ Will use ENSEMBLE for (k={gk[0]}, m={gk[1]})")
        else:
            print(f"\n→ Will keep ORIGINAL expert for (k={gk[0]}, m={gk[1]})")

In [ ]:
# ============================================================
# Cell 11: Save MoE Model (with Ensemble for Weak Groups)
# ============================================================

# Safe fallbacks if tuning cells (10b, 10c) were skipped
_use_ensemble = use_ensemble_for if 'use_ensemble_for' in dir() else []
_ensemble_size = ENSEMBLE_SIZE if 'ENSEMBLE_SIZE' in dir() else 0
_weak_ensembles = weak_ensembles if 'weak_ensembles' in dir() else {}
_weak_scalers = weak_scalers if 'weak_scalers' in dir() else {}
_expert_hparams = expert_hparams if 'expert_hparams' in dir() else {}

save_dict = {
    'feature_dim': FEATURE_DIM,
    'group_keys': group_keys,
    'ensemble_groups': _use_ensemble,
    'ensemble_size': _ensemble_size,
}

# Save original experts + scalers + architecture params for all groups
for gk in group_keys:
    key_str = f"k{gk[0]}_m{gk[1]}"
    save_dict[f'expert_{key_str}_state'] = experts[gk].cpu().state_dict()
    experts[gk].to(device)
    save_dict[f'scaler_{key_str}'] = group_data[gk]['scaler']
    # Save architecture params so we can reconstruct the model
    if gk in _expert_hparams:
        h = _expert_hparams[gk]
        save_dict[f'expert_{key_str}_hidden_dim'] = h[0]
        save_dict[f'expert_{key_str}_num_blocks'] = h[1]
        save_dict[f'expert_{key_str}_dropout'] = h[2]

# Save ensemble models + scalers for weak groups (if they exist)
for gk in _use_ensemble:
    if gk in _weak_ensembles:
        key_str = f"k{gk[0]}_m{gk[1]}"
        for ens_idx, ens_model in enumerate(_weak_ensembles[gk]):
            save_dict[f'ensemble_{key_str}_model{ens_idx}_state'] = ens_model.cpu().state_dict()
            ens_model.to(device)
        save_dict[f'ensemble_scaler_{key_str}'] = _weak_scalers[gk]
        # Save ensemble architecture configs
        ENSEMBLE_CONFIGS = [
            (512, 5, 0.08), (512, 6, 0.10), (384, 5, 0.08),
            (512, 4, 0.12), (640, 5, 0.10),
        ]
        for ci, (h, b, d) in enumerate(ENSEMBLE_CONFIGS[:_ensemble_size]):
            save_dict[f'ensemble_{key_str}_model{ci}_hidden_dim'] = h
            save_dict[f'ensemble_{key_str}_model{ci}_num_blocks'] = b
            save_dict[f'ensemble_{key_str}_model{ci}_dropout'] = d

torch.save(save_dict, 'moe_model.pth')

file_size = os.path.getsize('moe_model.pth') / 1024
print(f"Model saved: moe_model.pth ({file_size:.1f} KB)")
print(f"Contains {len(group_keys)} base expert DNNs")
if _use_ensemble:
    print(f"Contains {len(_use_ensemble)} ensemble groups × {_ensemble_size} models each")
    print(f"Ensemble groups: {_use_ensemble}")
else:
    print("No ensemble models (tuning cells were skipped or not needed).")

In [ ]:
# ============================================================
# Cell 12: Generate Predictions & Save (with Ensemble support)
# ============================================================

test_file_path = 'CSCE-636-Project-1-Train-n_k_m_P'  # <-- Change to test file
output_file_path = 'predicted_mHeights'

with open(test_file_path, 'rb') as f:
    test_data = pickle.load(f)
print(f"Loaded {len(test_data)} test samples.")

# Safely check if ensemble variables exist (in case tuning cells were skipped)
_use_ensemble = use_ensemble_for if 'use_ensemble_for' in dir() else []
_weak_ensembles = weak_ensembles if 'weak_ensembles' in dir() else {}
_weak_scalers = weak_scalers if 'weak_scalers' in dir() else {}

if _use_ensemble:
    print(f"Ensemble groups: {_use_ensemble}")
else:
    print("No ensemble — using original experts for all groups.")

# Group by (k, m)
test_groups = defaultdict(list)
for i, sample in enumerate(test_data):
    n, k, m, P = sample
    feat = extract_features(sample)
    test_groups[(k, m)].append((i, feat))

# Predict
predicted_mheights = [0.0] * len(test_data)

for gk, items in test_groups.items():
    indices = [item[0] for item in items]
    features = np.array([item[1] for item in items], dtype=np.float32)

    if gk not in experts:
        print(f"  WARNING: No expert for (k={gk[0]}, m={gk[1]}), default=1.0")
        for idx, _ in items:
            predicted_mheights[idx] = 1.0
        continue

    # Always get the original expert prediction
    scaler = group_data[gk]['scaler']
    features_scaled = scaler.transform(features)
    tensor = torch.tensor(features_scaled, dtype=torch.float32).to(device)

    model = experts[gk]
    model.eval()
    with torch.no_grad():
        all_preds = []
        for bi in range(0, len(tensor), 1024):
            batch = tensor[bi:bi+1024]
            pred = model(batch).cpu().numpy().flatten()
            all_preds.append(pred)
        orig_preds_log2 = np.concatenate(all_preds)

    # Check if this group uses ensemble
    if gk in _use_ensemble and gk in _weak_ensembles:
        # Average: original expert + all ensemble models
        all_pred_arrays = [orig_preds_log2]
        
        # Ensemble models use the same scaler (from group_data)
        ens_scaler = _weak_scalers.get(gk, scaler)
        ens_features_scaled = ens_scaler.transform(features)
        ens_tensor = torch.tensor(ens_features_scaled, dtype=torch.float32).to(device)

        for ens_model in _weak_ensembles[gk]:
            ens_model.eval()
            with torch.no_grad():
                batch_preds = []
                for bi in range(0, len(ens_tensor), 1024):
                    batch = ens_tensor[bi:bi+1024]
                    pred = ens_model(batch).cpu().numpy().flatten()
                    batch_preds.append(pred)
                all_pred_arrays.append(np.concatenate(batch_preds))

        preds_log2 = np.mean(all_pred_arrays, axis=0)
        source = f"ens({len(all_pred_arrays)})"
    else:
        preds_log2 = orig_preds_log2
        source = "single"

    preds = np.maximum(2.0 ** preds_log2, 1.0)
    for idx, val in zip(indices, preds):
        predicted_mheights[idx] = float(val)

    print(f"  (k={gk[0]},m={gk[1]}) [{source:>8}]: {len(indices)} samples, "
          f"range=[{preds.min():.4f}, {preds.max():.4f}]")

with open(output_file_path, 'wb') as f:
    pickle.dump(predicted_mheights, f)

print(f"\n✅ Predictions saved to '{output_file_path}'")
print(f"Total: {len(predicted_mheights)}, Range: [{min(predicted_mheights):.4f}, {max(predicted_mheights):.4f}]")
print(f"All >= 1: {all(h >= 1.0 for h in predicted_mheights)}")

Loaded 96524 test samples.
  (k=6,m=3): 10913 samples, range=[115.7867, 249211.5625]
  (k=4,m=3): 11003 samples, range=[5.0799, 4199.9590]
  (k=4,m=4): 10253 samples, range=[11.5932, 47669.0977]
  (k=5,m=2): 10803 samples, range=[4.0270, 6666.4741]
  (k=6,m=2): 18898 samples, range=[7.9266, 21594.0312]
  (k=5,m=3): 10159 samples, range=[9.9409, 10942.2412]
  (k=4,m=2): 11157 samples, range=[2.7875, 264.8725]
  (k=4,m=5): 6765 samples, range=[326.5062, 345844.7500]
  (k=5,m=4): 6573 samples, range=[297.4944, 182112.9531]

✅ Predictions saved to 'predicted_mHeights'
Total: 96524, Range: [2.7875, 345844.7500]
All >= 1: True


In [13]:
# ============================================================
# Cell 13: Download results from Colab
# ============================================================
from google.colab import files

print("Downloading model and predictions...")
files.download('moe_model.pth')
files.download('predicted_mHeights')
print("Done! Check your Downloads folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Check your Downloads folder.
